# Herramientas y visualización de datos - Actividad práctica final
## Visualización de grandes volúmenes de datos con Apache Spark

**Dataset:** MovieLens 32M + MovieLens Latest  
**Tema:** Análisis de calificaciones de películas mediante Spark  
**Integrantes:** Diego Bueno / JuanFer  
**Fecha:** 2026

## 1. Introducción

Se seleccionó MovieLens porque contiene millones de calificaciones reales de películas realizadas por usuarios. El dataset permite aplicar Apache Spark para cargar, limpiar, transformar y visualizar grandes volúmenes de datos.

El análisis responde preguntas sobre distribución de calificaciones, películas más populares, evolución temporal y relación entre popularidad y calificación promedio.

In [ ]:
# !pip install pyspark pandas matplotlib seaborn plotly

## 2. Configuración inicial de Spark

In [ ]:
import os
import sys
import tempfile
from pathlib import Path

if sys.platform.startswith('win'):
    hadoop_home = os.path.join(tempfile.gettempdir(), 'hadoop')
    os.makedirs(os.path.join(hadoop_home, 'bin'), exist_ok=True)
    os.environ['HADOOP_HOME'] = hadoop_home

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

spark = (
    SparkSession.builder
    .appName("MovieLens Big Data Visualization")
    .master("local[2]")
    .config("spark.driver.memory", "3g")
    .config("spark.sql.shuffle.partitions", "4")
    .getOrCreate()
)

print("Spark iniciado correctamente")

## 3. Rutas del dataset

In [ ]:
ruta_base = Path(r"C:\Users\JuanFer\Documents\Actividad 3")

ml32_path = ruta_base / "ml-32m" / "ml-32m"
mllatest_path = ruta_base / "ml-latest" / "ml-latest"

print("Existe ml-32m:", ml32_path.exists())
print("Existe ml-latest:", mllatest_path.exists())

print("Archivos ml-32m:")
print(list(ml32_path.glob("*")))

print("Archivos ml-latest:")
print(list(mllatest_path.glob("*")))

## 4. Verificación del tamaño del dataset

In [ ]:
import os

archivos_dataset = []
for carpeta in [ml32_path, mllatest_path]:
    if carpeta.exists():
        archivos_dataset.extend(list(carpeta.glob("*.csv")))

total_gb = 0
for archivo in archivos_dataset:
    size_gb = os.path.getsize(archivo) / (1024 ** 3)
    total_gb += size_gb
    print(f"{archivo.name}: {size_gb:.2f} GB")

print(f"\nTamaño total aproximado del dataset usado: {total_gb:.2f} GB")

if total_gb >= 1:
    print("El dataset cumple el requisito mínimo de 1 GB.")
else:
    print("Advertencia: el dataset no alcanza 1 GB. Verifica que estén las carpetas completas.")

## 5. Carga del dataset con Spark

In [ ]:
ratings_path = ml32_path / "ratings.csv"
movies_path = ml32_path / "movies.csv"
tags_path = ml32_path / "tags.csv"

ratings = spark.read.csv(str(ratings_path), header=True, inferSchema=True)
movies = spark.read.csv(str(movies_path), header=True, inferSchema=True)
tags = spark.read.csv(str(tags_path), header=True, inferSchema=True)

print("Schema de ratings:")
ratings.printSchema()

print("Schema de movies:")
movies.printSchema()

print("Muestra de ratings:")
ratings.limit(5).show()

print("Muestra de movies:")
movies.limit(5).show(truncate=False)

## 6. Conteo de registros

In [ ]:
total_ratings = ratings.count()
total_movies = movies.count()
total_tags = tags.count()

print(f"Total de registros en ratings: {total_ratings:,}")
print(f"Total de registros en movies: {total_movies:,}")
print(f"Total de registros en tags: {total_tags:,}")

## 7. Exploración inicial y limpieza

In [ ]:
nulls_ratings = ratings.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in ratings.columns
])

print("Valores nulos en ratings:")
nulls_ratings.show()

nulls_movies = movies.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in movies.columns
])

print("Valores nulos en movies:")
nulls_movies.show()

duplicados_ratings = ratings.count() - ratings.dropDuplicates().count()
duplicados_movies = movies.count() - movies.dropDuplicates().count()

print(f"Duplicados en ratings: {duplicados_ratings:,}")
print(f"Duplicados en movies: {duplicados_movies:,}")

ratings.select(
    F.min("rating").alias("rating_minimo"),
    F.max("rating").alias("rating_maximo"),
    F.avg("rating").alias("rating_promedio")
).show()

### Decisiones de limpieza

Se eliminan duplicados, registros nulos en columnas clave, calificaciones fuera del rango válido y se transforma el `timestamp` a fecha para análisis temporal.

In [ ]:
ratings_clean = (
    ratings
    .dropDuplicates()
    .filter(F.col("userId").isNotNull())
    .filter(F.col("movieId").isNotNull())
    .filter(F.col("rating").isNotNull())
    .filter(F.col("timestamp").isNotNull())
    .filter((F.col("rating") >= 0.5) & (F.col("rating") <= 5.0))
    .withColumn("fecha", F.to_date(F.from_unixtime(F.col("timestamp"))))
    .withColumn("anio", F.year(F.col("fecha")))
)

movies_clean = (
    movies
    .dropDuplicates()
    .filter(F.col("movieId").isNotNull())
    .filter(F.col("title").isNotNull())
    .filter(F.col("genres").isNotNull())
)

ratings_clean.cache()
movies_clean.cache()

print(f"Registros limpios en ratings: {ratings_clean.count():,}")
print(f"Registros limpios en movies: {movies_clean.count():,}")
ratings_clean.limit(5).show()

## 8. Integración de datos

In [ ]:
ratings_movies = ratings_clean.join(movies_clean, on="movieId", how="inner")
ratings_movies.limit(5).show(truncate=False)

## Visualización 1: Distribución de calificaciones

In [ ]:
rating_distribution = (
    ratings_clean
    .groupBy("rating")
    .count()
    .orderBy("rating")
    .toPandas()
)

plt.figure(figsize=(9,5))
sns.barplot(data=rating_distribution, x="rating", y="count", color="#4C78A8")
plt.title("Distribución de calificaciones en MovieLens")
plt.xlabel("Calificación")
plt.ylabel("Cantidad de registros")
plt.grid(axis="y", alpha=0.3)
plt.show()

**Interpretación:** La distribución muestra cuáles calificaciones son más frecuentes y permite observar la tendencia general de los usuarios al valorar películas.

## Visualización 2: Películas con mayor número de calificaciones

In [ ]:
top_movies = (
    ratings_movies
    .groupBy("title")
    .agg(F.count("rating").alias("cantidad_calificaciones"))
    .orderBy(F.desc("cantidad_calificaciones"))
    .limit(10)
    .toPandas()
)

plt.figure(figsize=(10,6))
sns.barplot(data=top_movies, y="title", x="cantidad_calificaciones", color="#59A14F")
plt.title("Top 10 películas con más calificaciones")
plt.xlabel("Cantidad de calificaciones")
plt.ylabel("Película")
plt.grid(axis="x", alpha=0.3)
plt.show()

**Interpretación:** Las películas con más calificaciones representan los títulos de mayor popularidad dentro del dataset.

## Visualización 3: Evolución de calificaciones por año

In [ ]:
ratings_by_year = (
    ratings_clean
    .filter(F.col("anio").isNotNull())
    .groupBy("anio")
    .count()
    .orderBy("anio")
    .toPandas()
)

plt.figure(figsize=(11,5))
sns.lineplot(data=ratings_by_year, x="anio", y="count", marker="o", color="#F28E2B")
plt.title("Evolución anual de calificaciones")
plt.xlabel("Año")
plt.ylabel("Cantidad de calificaciones")
plt.grid(alpha=0.3)
plt.show()

**Interpretación:** La tendencia temporal permite identificar periodos de mayor o menor actividad de los usuarios.

## Visualización 4: Popularidad vs calificación promedio

In [ ]:
movie_stats = (
    ratings_movies
    .groupBy("title")
    .agg(
        F.count("rating").alias("cantidad_calificaciones"),
        F.avg("rating").alias("promedio_rating")
    )
    .filter(F.col("cantidad_calificaciones") >= 500)
    .orderBy(F.desc("cantidad_calificaciones"))
    .limit(1000)
    .toPandas()
)

plt.figure(figsize=(9,6))
sns.scatterplot(
    data=movie_stats,
    x="cantidad_calificaciones",
    y="promedio_rating",
    alpha=0.6,
    color="#E15759"
)
plt.title("Relación entre popularidad y calificación promedio")
plt.xlabel("Cantidad de calificaciones")
plt.ylabel("Promedio de rating")
plt.grid(alpha=0.3)
plt.show()

**Interpretación:** La visualización permite analizar si las películas más populares también tienen mejores promedios de calificación.

## Visualización 5: Mapa de calor de calificaciones por año

In [ ]:
heatmap_data = (
    ratings_clean
    .filter(F.col("anio").isNotNull())
    .groupBy("anio", "rating")
    .count()
    .orderBy("anio", "rating")
    .toPandas()
)

heatmap_data = heatmap_data[heatmap_data["anio"] >= 2000]

pivot_heatmap = heatmap_data.pivot_table(
    index="anio",
    columns="rating",
    values="count",
    fill_value=0
)

plt.figure(figsize=(10,8))
sns.heatmap(pivot_heatmap, cmap="Blues")
plt.title("Mapa de calor: cantidad de calificaciones por año y rating")
plt.xlabel("Rating")
plt.ylabel("Año")
plt.show()

**Interpretación:** El mapa de calor permite observar la concentración de calificaciones por año y por valor de rating.

## 9. Conclusiones

Apache Spark permitió cargar y procesar un dataset de gran volumen sin convertir directamente todos los datos a Pandas.  
Los hallazgos muestran la distribución de calificaciones, las películas más populares, la evolución temporal de la actividad y la relación entre popularidad y promedio de rating.

### Limitaciones

El rendimiento depende del equipo local. Por eso, todas las agregaciones pesadas se hacen con Spark y solo los resultados reducidos se convierten a Pandas.

## 10. Cierre de Spark

In [ ]:
spark.stop()
print("Sesión Spark cerrada correctamente")